# 09 Ablation Experiments

This notebook organizes the ablation analysis for the project. It runs executable structured-feature ablations immediately and incorporates saved multimodal fusion results when those training artifacts are available.

## Ablation scope

- Vitals only
- Vitals plus labs
- Full structured EHR
- Planned text-only comparison
- Multimodal fusion comparison when Notebook 07 artifacts exist

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost

In [3]:
import pandas as pd

from src.evaluation.ablation import (
    build_fusion_strategy_table,
    build_planned_ablation_matrix,
    run_structured_ablation_suite,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [4]:
config['ablation']

{'executable_variants': ['vitals_only', 'vitals_labs', 'structured_full'],
 'planned_variants': ['vitals_only',
  'vitals_labs',
  'structured_full',
  'text_only',
  'multimodal_fusion'],
 'baseline_models': ['logistic_regression']}

## Load horizon datasets and multimodal experiment plan

In [5]:
horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing structured horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'], low_memory=False)

multimodal_dir = paths['processed_data_dir'] / '07_multimodal_models'
multimodal_plan_files = sorted(multimodal_dir.glob('*_multimodal_experiment_plan.csv'))
multimodal_result_files = sorted(multimodal_dir.glob('*_multimodal_results.csv'))
multimodal_plan_df = pd.concat([pd.read_csv(path) for path in multimodal_plan_files], ignore_index=True) if multimodal_plan_files else pd.DataFrame()
multimodal_results_df = pd.concat([pd.read_csv(path) for path in multimodal_result_files], ignore_index=True) if multimodal_result_files else pd.DataFrame()
{key: value.shape for key, value in horizon_tables.items()}, multimodal_plan_df.shape, multimodal_results_df.shape

({'horizon_6h': (1268294, 90),
  'horizon_12h': (1131670, 90),
  'horizon_24h': (860480, 90)},
 (12, 9),
 (24, 21))

## Run executable structured ablations

In [6]:
artifacts = run_structured_ablation_suite(horizon_tables, config)
ablation_results_df = artifacts['ablation_results']
ablation_results_df

,dataset_name,variant_name,model_name,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,n_features,n_examples
0,horizon_6h,vitals_only,logistic_regression,0.706446,0.121689,0.099595,0.575251,0.169793,0.204089,0.718705,0.575251,172,1555,3973,127,128,5827
1,horizon_6h,vitals_labs,logistic_regression,0.712491,0.121196,0.103428,0.595318,0.176238,0.199540,0.720876,0.595318,178,1543,3985,121,288,5827
2,horizon_6h,structured_full,logistic_regression,0.844136,0.243081,0.165198,0.752508,0.270921,0.154118,0.794320,0.752508,225,1137,4391,74,304,5827
3,horizon_12h,vitals_only,logistic_regression,0.726444,0.131325,0.091168,0.669456,0.160481,0.209836,0.698374,0.669456,160,1595,3693,79,128,5527
4,horizon_12h,vitals_labs,logistic_regression,0.719244,0.123234,0.089057,0.640167,0.156362,0.203008,0.704047,0.640167,153,1565,3723,86,288,5527
5,horizon_12h,structured_full,logistic_regression,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48,304,5527
6,horizon_24h,vitals_only,logistic_regression,0.694471,0.088949,0.075000,0.668571,0.134870,0.217013,0.624512,0.668571,117,1443,2400,58,128,4018
7,horizon_24h,vitals_labs,logistic_regression,0.679246,0.094324,0.074728,0.628571,0.133576,0.208247,0.645589,0.628571,110,1362,2481,65,288,4018
8,horizon_24h,structured_full,logistic_regression,0.897076,0.339245,0.179975,0.811429,0.294606,0.116882,0.831642,0.811429,142,647,3196,33,304,4018


## Build planned ablation matrix and fusion strategy table

In [7]:
planned_matrix_df = build_planned_ablation_matrix(config)
fusion_source_df = multimodal_results_df if not multimodal_results_df.empty else multimodal_plan_df
fusion_strategy_df = build_fusion_strategy_table(fusion_source_df)
planned_matrix_df, fusion_strategy_df

(        variant_name                                        description  \
 0        vitals_only                      Structured vitals subset only   
 1        vitals_labs                      Vitals plus laboratory subset   
 2    structured_full                        All structured EHR features   
 3          text_only         Clinical notes without structured features   
 4  multimodal_fusion  Structured plus text with multiple fusion stra...   
 
    implemented_now  
 0             True  
 1             True  
 2             True  
 3            False  
 4            False  ,
    structured_encoder dataset_name split     auprc     auroc      loss  \
 0                 gru  horizon_12h  test  0.200648  0.783415  0.192318   
 1                 gru  horizon_12h  test  0.219549  0.788869  0.176303   
 2                 gru  horizon_12h  test  0.176338  0.780048  0.159550   
 3                 gru  horizon_12h  test  0.064832  0.618505  0.254155   
 4                 gru  horizon_12

## Inspect strongest ablation results

In [8]:
ablation_results_df.sort_values('auprc', ascending=False).head(12) if not ablation_results_df.empty else ablation_results_df

,dataset_name,variant_name,model_name,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,n_features,n_examples
8,horizon_24h,structured_full,logistic_regression,0.897076,0.339245,0.179975,0.811429,0.294606,0.116882,0.831642,0.811429,142,647,3196,33,304,4018
5,horizon_12h,structured_full,logistic_regression,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48,304,5527
2,horizon_6h,structured_full,logistic_regression,0.844136,0.243081,0.165198,0.752508,0.270921,0.154118,0.794320,0.752508,225,1137,4391,74,304,5827
3,horizon_12h,vitals_only,logistic_regression,0.726444,0.131325,0.091168,0.669456,0.160481,0.209836,0.698374,0.669456,160,1595,3693,79,128,5527
4,horizon_12h,vitals_labs,logistic_regression,0.719244,0.123234,0.089057,0.640167,0.156362,0.203008,0.704047,0.640167,153,1565,3723,86,288,5527
0,horizon_6h,vitals_only,logistic_regression,0.706446,0.121689,0.099595,0.575251,0.169793,0.204089,0.718705,0.575251,172,1555,3973,127,128,5827
1,horizon_6h,vitals_labs,logistic_regression,0.712491,0.121196,0.103428,0.595318,0.176238,0.199540,0.720876,0.595318,178,1543,3985,121,288,5827
7,horizon_24h,vitals_labs,logistic_regression,0.679246,0.094324,0.074728,0.628571,0.133576,0.208247,0.645589,0.628571,110,1362,2481,65,288,4018
6,horizon_24h,vitals_only,logistic_regression,0.694471,0.088949,0.075000,0.668571,0.134870,0.217013,0.624512,0.668571,117,1443,2400,58,128,4018


## Save ablation artifacts

In [9]:
output_dir = paths['processed_data_dir'] / '09_ablation_experiments'
artifact_bundle = dict(artifacts)
artifact_bundle['planned_ablation_matrix'] = planned_matrix_df
artifact_bundle['fusion_strategy_table'] = fusion_strategy_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'horizon_6h_vitals_only_tabular': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/09_ablation_experiments/horizon_6h_vitals_only_tabular.csv',
 'horizon_6h_vitals_only_logistic_regression_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/09_ablation_experiments/horizon_6h_vitals_only_logistic_regression_predictions.csv',
 'horizon_6h_vitals_labs_tabular': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/09_ablation_experiments/horizon_6h_vitals_labs_tabular.csv',
 'horizon_6h_vitals_labs_logistic_regression_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/09_ablation_experiments/horizon_6h_vitals_labs_logistic_regression_predictions.csv',
 'horizon_6h_structured_full_tabular': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/09_ablation_experiments/horizon_6h_structured_full_tabular.csv',
 'horizon_6h_structured_full_logistic_regression_predict

In [10]:
manifest_path = paths['manifests_dir'] / '09_ablation_experiments_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='09_ablation_experiments',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'multimodal_plan_available': bool(len(multimodal_plan_df) or len(multimodal_results_df)),
        'ablation_result_rows': int(len(ablation_results_df)),
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/09_ablation_experiments_manifest.json')

## Review checklist

Before explainability analysis, review:

- How much performance comes from vitals alone?
- Do labs add meaningful signal beyond vitals?
- Which horizons benefit most from richer structured features?
- Which fusion strategies should be prioritized once full multimodal training outputs are available?
- Is the planned ablation matrix complete enough for the paper?

## Next notebook

`10_explainability.ipynb` will prepare SHAP-based structured explanations, attention-based note interpretation, and clinically meaningful narratives for the final report.